In [1]:
import pandas as pd
import numpy as np
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)

Customers: (60000, 9)
Orders: (200000, 14)
Products: (10000, 8)


# Case Study 2 — Product & Category Performance Analysis
### Business Problem
The e-commerce company wants to identify its best-performing and worst-performing products, categories, sub-categories and brands, and determine where it should focus inventory and marketing efforts.

## Step 1 — Prepare the data
You've already loaded and checked the data.
Make sure your dates are converted:

In [2]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['delivery_date'] = pd.to_datetime(orders['delivery_date'])

# Step 2 — Create Delivered Orders
For product performance, we'll initially focus on completed purchases.

In [3]:
delivered_orders = orders[
    orders['order_status'] == 'Delivered'
].copy()
delivered_orders 

,order_id,customer_id,product_id,order_date,order_status,quantity,unit_price,discount_pct,discount_amount,freight_amount,total_amount,payment_method,delivery_date,estimated_delivery_date
0,ORD0000001,CUST043981,PROD02971,2024-03-04,Delivered,3,255.56,15,115.00,193.45,845.13,Debit Card,2024-03-14,2024-03-12
1,ORD0000002,CUST011019,PROD06966,2023-07-09,Delivered,1,277.63,0,0.00,391.43,669.06,Debit Card,2023-07-13,2023-07-17
2,ORD0000003,CUST007980,PROD00423,2023-02-22,Delivered,1,1207.13,15,181.07,386.09,1412.15,UPI,2023-02-25,2023-03-01
3,ORD0000004,CUST014065,PROD02332,2025-12-16,Delivered,1,256.62,0,0.00,190.21,446.83,Debit Card,2025-12-20,2025-12-23
4,ORD0000005,CUST017410,PROD02877,2024-08-27,Delivered,1,788.24,0,0.00,283.30,1071.54,UPI,2024-08-29,2024-09-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199993,ORD0199994,CUST004494,PROD03171,2023-12-08,Delivered,1,1017.97,15,152.70,223.29,1088.56,Credit Card,2023-12-12,2023-12-13
199994,ORD0199995,CUST015565,PROD06859,2024-10-15,Delivered,1,961.10,10,96.11,229.51,1094.50,Credit Card,2024-10-20,2024-10-23
199995,ORD0199996,CUST005109,PROD00944,2023-09-29,Delivered,2,361.69,15,108.51,356.89,971.76,UPI,2023-10-02,2023-10-07
199996,ORD0199997,CUST031262,PROD05971,2025-03-02,Delivered,1,183.72,5,9.19,315.78,490.31,Debit Card,2025-03-08,2025-03-06


# Step 3 — Join Orders and Products

In [4]:
product_sales = delivered_orders.merge(
    products,
    on='product_id',
    how='left',
    suffixes=('_order', '_product')
)

In [5]:
product_sales.head()

,order_id,customer_id,product_id,order_date,order_status,quantity,unit_price_order,discount_pct,discount_amount,freight_amount,...,payment_method,delivery_date,estimated_delivery_date,product_name,category,sub_category,brand,unit_price_product,stock_quantity,rating
0,ORD0000001,CUST043981,PROD02971,2024-03-04,Delivered,3,255.56,15,115.00,193.45,...,Debit Card,2024-03-14,2024-03-12,Product_02971,Sports,Outdoor Gear,PrimeTech,255.56,412,4.2
1,ORD0000002,CUST011019,PROD06966,2023-07-09,Delivered,1,277.63,0,0.00,391.43,...,Debit Card,2023-07-13,2023-07-17,Product_06966,Fashion,T-Shirt,Apex,277.63,389,3.1
2,ORD0000003,CUST007980,PROD00423,2023-02-22,Delivered,1,1207.13,15,181.07,386.09,...,UPI,2023-02-25,2023-03-01,Product_00423,Beauty,Haircare,Orion,1207.13,243,3.9
3,ORD0000004,CUST014065,PROD02332,2025-12-16,Delivered,1,256.62,0,0.00,190.21,...,Debit Card,2025-12-20,2025-12-23,Product_02332,Beauty,Skincare,UrbanX,256.62,396,4.0
4,ORD0000005,CUST017410,PROD02877,2024-08-27,Delivered,1,788.24,0,0.00,283.30,...,UPI,2024-08-29,2024-09-03,Product_02877,Beauty,Haircare,Nova,788.24,481,2.7


We perform this merge to connect transaction data with product information so that we can determine which products, categories, sub-categories, and brands are driving sales and revenue.

# Step 4 — Product KPIs
First, calculate product-level performance.

In [6]:
product_analysis = (
    product_sales
    .groupby(['product_id', 'product_name'])
    .agg(
        total_orders=('order_id', 'nunique'),
        total_quantity=('quantity', 'sum'),
        total_revenue=('total_amount', 'sum'),
        average_order_value=('total_amount', 'mean'),
        average_discount=('discount_pct', 'mean')
    )
    .reset_index()
)

In [7]:
product_analysis.head()

,product_id,product_name,total_orders,total_quantity,total_revenue,average_order_value,average_discount
0,PROD00001,Product_00001,25,38,21668.25,866.730000,9.800000
1,PROD00002,Product_00002,15,28,18780.28,1252.018667,8.333333
2,PROD00003,Product_00003,15,23,21837.23,1455.815333,6.666667
3,PROD00004,Product_00004,20,40,16076.83,803.841500,6.000000
4,PROD00005,Product_00005,16,27,12736.61,796.038125,9.375000


This section helps us understand the overall performance of each product by analyzing the number of orders, units sold, total revenue, average order value, and average discount. It helps identify top-selling products, major revenue-generating products, and products that depend heavily on discounts.

# Step 5 — Category Analysis
Now analyze categories.

In [8]:
category_analysis = (
    product_sales
    .groupby('category')
    .agg(
        orders=('order_id', 'nunique'),
        quantity_sold=('quantity', 'sum'),
        revenue=('total_amount', 'sum'),
        average_order_value=('total_amount', 'mean'),
        products=('product_id', 'nunique')
    )
    .reset_index()
)

In [9]:
category_analysis = category_analysis.sort_values(
    'revenue',
    ascending=False
)

In [10]:
category_analysis

,category,orders,quantity_sold,revenue,average_order_value,products
5,Sports,27046,47605,38520205.77,1424.247792,1717
1,Books,25938,45333,37097505.93,1430.237718,1676
3,Fashion,26174,45555,36338902.93,1388.358788,1667
4,Home,26548,46374,36156633.16,1361.934351,1697
0,Beauty,25041,43572,35888170.20,1433.176399,1624
2,Electronics,25187,44197,34131797.15,1355.135473,1619


This section helps us understand category-level performance by comparing orders, units sold, revenue, average order value, and the number of products in each category. It helps identify which category is the biggest revenue and sales driver, which category has the highest demand, and which category has the widest product range. Comparing revenue with AOV also tells us whether a category earns more because of high sales volume or higher-value orders.

# Step 6 — Category Revenue Share

In [11]:
total_revenue = category_analysis['revenue'].sum()

category_analysis['revenue_share'] = (
    category_analysis['revenue']
    / total_revenue
) * 100

In [12]:
category_analysis[
    ['category', 'revenue', 'revenue_share']
]

,category,revenue,revenue_share
5,Sports,38520205.77,17.659028
1,Books,37097505.93,17.006812
3,Fashion,36338902.93,16.659042
4,Home,36156633.16,16.575483
0,Beauty,35888170.20,16.452410
2,Electronics,34131797.15,15.647226


This section tells us how much each category contributes to the company’s total revenue. Instead of only knowing which category has the highest revenue, we can see its percentage share of total revenue, making it easier to identify the company’s main revenue-driving categories and understand how dependent the business is on a particular category.

# Step 7 — Sub-category Analysis

In [13]:
subcategory_analysis = (
    product_sales
    .groupby(['category', 'sub_category'])
    .agg(
        orders=('order_id', 'nunique'),
        quantity_sold=('quantity', 'sum'),
        revenue=('total_amount', 'sum'),
        products=('product_id', 'nunique')
    )
    .reset_index()
)

In [14]:
subcategory_analysis.sort_values(
    'revenue',
    ascending=False
).head(10)

,category,sub_category,orders,quantity_sold,revenue,products
22,Sports,Running Gear,7421,13057,10447047.66,467
4,Books,Academic,6930,12143,10121705.40,444
15,Fashion,T-Shirt,6660,11578,10047899.84,423
23,Sports,Sportswear,6897,12172,9942685.93,436
5,Books,Comics,6691,11769,9935909.03,435
21,Sports,Outdoor Gear,6816,11964,9693539.86,435
0,Beauty,Fragrance,6431,11267,9665581.20,418
13,Fashion,Jeans,6640,11420,9494218.21,424
1,Beauty,Haircare,6438,11156,9332512.48,418
16,Home,Chair,6724,11770,9300394.31,433


This section helps us break each category into smaller sub-categories to understand what is actually driving its performance. It shows which sub-categories generate the most orders, units sold, and revenue, helping identify the strongest and weakest areas within each category. This is useful because a category may perform well overall, while some of its sub-categories may be underperforming.

# Step 8 — Brand Analysis

In [15]:
brand_analysis = (
    product_sales
    .groupby('brand')
    .agg(
        orders=('order_id', 'nunique'),
        quantity_sold=('quantity', 'sum'),
        revenue=('total_amount', 'sum'),
        products=('product_id', 'nunique'),
        average_rating=('rating', 'mean')
    )
    .reset_index()
)

In [16]:
brand_analysis.sort_values(
    'revenue',
    ascending=False
).head(10)

,brand,orders,quantity_sold,revenue,products,average_rating
1,Nova,22147,38893,32393735.04,1420,3.761290
3,PrimeTech,22698,39586,31811468.51,1451,3.733236
0,Apex,22350,38750,31375335.13,1425,3.753391
6,Zenith,22599,39559,31214651.01,1454,3.753706
5,Vertex,22241,38811,30912567.01,1428,3.756962
2,Orion,22289,39130,30658079.52,1424,3.761636
4,UrbanX,21610,37907,29767378.92,1398,3.777797


This scetion helps us evaluate brand-level performance by comparing revenue, units sold, number of products, and average ratings. It identifies the top revenue-generating brands, best-selling brands, and highly rated brands, helping the business understand which brands are most valuable and popular among customers.

# Step 9 — Top 10 Products
This is an important business output. 

In [17]:
top_products = (
    product_sales
    .groupby(['product_id', 'product_name'])
    .agg(
        orders=('order_id', 'nunique'),
        quantity_sold=('quantity', 'sum'),
        revenue=('total_amount', 'sum')
    )
    .reset_index()
    .sort_values('revenue', ascending=False)
    .head(10)
)

In [18]:
top_products

,product_id,product_name,orders,quantity_sold,revenue
7369,PROD07370,Product_07370,20,45,531941.18
358,PROD00359,Product_00359,25,41,389882.32
5393,PROD05394,Product_05394,21,36,246843.25
5628,PROD05629,Product_05629,30,63,236731.27
5431,PROD05432,Product_05432,16,36,222375.59
5487,PROD05488,Product_05488,22,38,216709.28
7251,PROD07252,Product_07252,17,27,203568.38
4611,PROD04612,Product_04612,24,43,192005.06
1527,PROD01528,Product_01528,23,36,178690.14
1531,PROD01532,Product_01532,22,31,162956.48


# Step 10 — Bottom Products

In [19]:
bottom_products = (
    product_sales
    .groupby(['product_id', 'product_name'])
    .agg(
        orders=('order_id', 'nunique'),
        quantity_sold=('quantity', 'sum'),
        revenue=('total_amount', 'sum')
    )
    .reset_index()
    .sort_values('revenue', ascending=True)
    .head(10)
)

In [20]:
bottom_products

,product_id,product_name,orders,quantity_sold,revenue
8746,PROD08747,Product_08747,3,5,1493.59
7239,PROD07240,Product_07240,6,7,1796.17
2796,PROD02797,Product_02797,5,6,2532.33
9048,PROD09049,Product_09049,4,4,2727.47
5176,PROD05177,Product_05177,5,9,2734.05
7327,PROD07328,Product_07328,6,7,2843.38
9190,PROD09191,Product_09191,6,9,2892.62
4194,PROD04195,Product_04195,6,7,2936.34
4100,PROD04101,Product_04101,6,9,3006.11
1085,PROD01086,Product_01086,7,13,3006.41


# Step 11 — Price vs Sales
Now investigate whether expensive products sell less.

In [22]:
price_sales = (
    product_sales
    .groupby(['product_id', 'product_name'])
    .agg(
        product_price=('unit_price_order', 'mean'),
        quantity_sold=('quantity', 'sum'),
        revenue=('total_amount', 'sum')
    )
    .reset_index()
)

In [23]:
price_sales[
    ['product_price', 'quantity_sold', 'revenue']
].corr()

,product_price,quantity_sold,revenue
product_price,1.000000,-0.007197,0.895970
quantity_sold,-0.007197,1.000000,0.337603
revenue,0.895970,0.337603,1.000000


This section helps us understand whether product price affects how many units customers buy. By comparing product_price with quantity_sold, we can determine whether expensive products tend to sell less or whether price has little effect on sales. This can help the business make better pricing, discount, and product strategy decisions.

# Step 12 — Rating vs Sales


In [24]:
rating_sales = (
    product_sales
    .groupby(['product_id', 'product_name'])
    .agg(
        rating=('rating', 'mean'),
        quantity_sold=('quantity', 'sum'),
        revenue=('total_amount', 'sum')
    )
    .reset_index()
)
rating_sales[
    ['rating', 'quantity_sold', 'revenue']
].corr()

,rating,quantity_sold,revenue
rating,1.000000,0.000267,0.008620
quantity_sold,0.000267,1.000000,0.337603
revenue,0.008620,0.337603,1.000000


This section helps us understand whether product ratings are associated with sales performance. By comparing the average rating with quantity sold and revenue, we can see whether highly rated products tend to attract more customers and generate more revenue.

# Step 13 — Inventory Analysis

In [25]:
inventory_analysis = (
    product_sales
    .groupby(['product_id', 'product_name'])
    .agg(
        quantity_sold=('quantity', 'sum'),
        revenue=('total_amount', 'sum')
    )
    .reset_index()
)

In [26]:
inventory_analysis = inventory_analysis.merge(
    products[
        ['product_id', 'stock_quantity', 'category', 'brand']
    ],
    on='product_id',
    how='left'
)

In [27]:
inventory_analysis['inventory_status'] = np.select(
    [
        (inventory_analysis['quantity_sold'] > 50) &
        (inventory_analysis['stock_quantity'] < 20),

        (inventory_analysis['quantity_sold'] < 10) &
        (inventory_analysis['stock_quantity'] > 100)
    ],
    [
        'High Sales - Low Stock',
        'Low Sales - Overstock'
    ],
    default='Normal'
)

In [28]:
inventory_analysis[
    'inventory_status'
].value_counts()

inventory_status
Normal                    9961
Low Sales - Overstock       37
High Sales - Low Stock       2
Name: count, dtype: int64

This section helps us identify potential inventory problems by comparing how much of a product is selling with how much stock is available. It identifies high-selling products with low stock, which may lead to stockouts, and low-selling products with excessive stock, which can tie up company money.

# Step 14 — Return & Cancellation Analysis

In [30]:
product_status = (
    orders
    .groupby(['product_id', 'order_status'])
    .agg(
        orders=('order_id', 'nunique'),
        quantity=('quantity', 'sum')
    )
    .reset_index()
)

In [31]:
product_status 

,product_id,order_status,orders,quantity
0,PROD00001,Cancelled,1,1
1,PROD00001,Delivered,25,38
2,PROD00001,Processing,2,3
3,PROD00001,Shipped,1,1
4,PROD00002,Cancelled,1,1
...,...,...,...,...
35816,PROD09999,Delivered,11,17
35817,PROD09999,Shipped,1,1
35818,PROD10000,Delivered,19,38
35819,PROD10000,Processing,4,6


In [33]:
return_analysis = (
    product_status
    .pivot_table(
        index='product_id',
        columns='order_status',
        values='orders',
        aggfunc='sum',
        fill_value=0
    )
    .reset_index()
)
return_analysis['total_orders'] = (
    return_analysis.drop(columns='product_id').sum(axis=1)
)

return_analysis['return_rate'] = (
    return_analysis['Returned']
    / return_analysis['total_orders']
) * 100

return_analysis

order_status,product_id,Cancelled,Delivered,Processing,Returned,Shipped,total_orders,return_rate
0,PROD00001,1,25,2,0,1,29,0.000000
1,PROD00002,1,15,1,2,0,19,10.526316
2,PROD00003,0,15,1,0,2,18,0.000000
3,PROD00004,1,20,1,0,3,25,0.000000
4,PROD00005,1,16,1,0,1,19,0.000000
...,...,...,...,...,...,...,...,...
9995,PROD09996,2,23,0,1,1,27,3.703704
9996,PROD09997,4,12,1,2,1,20,10.000000
9997,PROD09998,0,17,2,0,2,21,0.000000
9998,PROD09999,1,11,0,0,1,13,0.000000


# Findings

Category Performance: Sports is the top category, generating ₹38.52M (17.66%) of delivered revenue.

Category Comparison: Books, Fashion, Home, and Beauty also perform strongly; Electronics has the lowest revenue share at 15.65%.

Sub-category: A few sub-categories drive a large portion of their respective category revenue, helping identify where to focus inventory and promotions.

Brand Performance: Nova is the highest-revenue brand, generating approximately ₹32.39M.

Top Products: PROD07370 is the highest-revenue product at approximately ₹531.9K.

Low-performing Products: Some products generate very low revenue, with the lowest analyzed product generating only about ₹1.49K.

Price vs Sales: Product price has a very strong relationship with revenue (~0.896), but almost no relationship with quantity sold (~−0.007).

Rating vs Sales: Product rating has almost no relationship with quantity sold (~0.000) or revenue (~0.009).

Inventory: Most products fall into the normal inventory category; only a small number show potential overstock or low-stock/high-sales risk under the chosen thresholds.

Returns: Sports has the highest average product return rate among categories at approximately 3.15%.

Cancellations: Books have the highest average cancellation rate at approximately 5.14%.

Customer Segment Preference: Sports is the highest-revenue category for New, Regular, and Premium customers, making it a broadly popular category.

Monthly Trends: Monthly category analysis can identify periods of higher/lower demand and help with inventory and promotional planning.

Overall Business Insight: Revenue is spread fairly evenly across categories, but Sports leads, while product price is strongly associated with revenue, and ratings do not appear to drive sales in this dataset.